# C2 classifier — DeBERTa-v3-base fine-tune on Colab

Tiny launcher. The actual training code lives in `mostargate/classifier/train.py` in the repo; this notebook just bootstraps a Colab T4 instance and runs that module.

**Before running:**
1. `Runtime → Change runtime type → T4 GPU` (free tier).
2. `Runtime → Run all`.
3. When done, download `classifier_model.zip` from the **Files** panel on the left and unzip into `dataset/classifier_artifacts/model/` locally.

Expected wall time on T4: ~10–15 minutes including model download. Training runs in fp32 by default — see `docs/phase_c_classifier_findings.md` §2 for the fp16 transformers 5.x regression context.

## 1. Pull the repo

In [ ]:
!git clone https://github.com/0xballistics/mostargate.git
%cd mostargate

## 2. Install Python deps

Colab images already include `torch` and `numpy`. We add the few packages `mostargate.classifier.train` imports beyond that.

In [ ]:
!pip install -q transformers datasets sentencepiece protobuf python-dotenv

## 3. Verify GPU is attached

In [ ]:
!nvidia-smi

## 4. Train

Hyperparameters are baked into `train.py` per `docs/phase_c_classifier_findings.md` §2 — 5 epochs, batch 16 (GPU), lr 2e-5, max_len 256, seed 42, BCEWithLogitsLoss with per-class pos_weight clipped at 10. No internal validation hold-out: all 500 records train the model, save the final checkpoint after epoch 5.

fp32 by default. If you have a transformers version that handles fp16 grad unscaling correctly, you can opt in with `--fp16` (faster but currently buggy on transformers 5.x).

The first run downloads `microsoft/deberta-v3-base` (~700 MB) from the HuggingFace hub. Subsequent re-runs in the same session use the cached download.

In [ ]:
!python -m mostargate.classifier.train

## 5. Zip the trained model for download

Right-click `classifier_model.zip` in the file panel on the left → `Download`. Unzip locally so the directory ends up at `dataset/classifier_artifacts/model/`.

In [ ]:
!zip -qr classifier_model.zip dataset/classifier_artifacts/model/
!ls -lh classifier_model.zip
print("\nDownload classifier_model.zip from the file panel (left sidebar).")